# Day 4 — Real-Time Monitoring & Alerting

## Objective

Develop a monitoring and alerting component for the predictive-maintenance system.

The trained Random Forest model will be used to estimate the probability of equipment failure from incoming sensor measurements.

The monitoring component will:
- Load the trained machine-learning model.
- Apply the same preprocessing used during model development.
- Accept equipment sensor measurements.
- Generate a failure probability.
- Assign an equipment risk level.
- Provide a maintenance recommendation.
- Prepare the prediction output for integration with the project dashboard.

This stage connects the machine-learning model to the operational predictive-maintenance workflow.

## 1. Load the Trained Model and Preprocessing Pipeline

The trained Random Forest model and the scaler created during the previous stages are loaded for use in the monitoring system.

Using the saved artifacts ensures that incoming sensor measurements are processed in the same way as the data used during model training.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

In [2]:
models_path = Path("../models")

model = joblib.load(
    models_path / "random_forest_model.joblib"
)

scaler = joblib.load(
    models_path / "scaler.joblib"
)

print("Model and scaler loaded successfully.")

Model and scaler loaded successfully.


## 2. Define Sensor Inputs

The monitoring system uses the same five numerical sensor measurements and machine type used during model development.

The identifier `Product ID` and the failure-mode indicators are not used because they were excluded during model development.

In [3]:
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

feature_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Type_L",
    "Type_M"
]

In [4]:
sample_machine = {
    "Type": "M",
    "Air temperature [K]": 298.1,
    "Process temperature [K]": 308.6,
    "Rotational speed [rpm]": 1551,
    "Torque [Nm]": 42.8,
    "Tool wear [min]": 0
}

sample_machine

{'Type': 'M',
 'Air temperature [K]': 298.1,
 'Process temperature [K]': 308.6,
 'Rotational speed [rpm]': 1551,
 'Torque [Nm]': 42.8,
 'Tool wear [min]': 0}

In [6]:
input_df = pd.DataFrame([sample_machine])

input_df["Type_L"] = (input_df["Type"] == "L").astype(int)
input_df["Type_M"] = (input_df["Type"] == "M").astype(int)

input_df = input_df.drop(columns=["Type"])

input_df = input_df.reindex(
    columns=feature_columns,
    fill_value=0
)

input_df

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,1


## 3. Preprocess Incoming Sensor Data

The numerical sensor measurements are transformed using the scaler fitted during model development.

The categorical machine-type indicators are already represented as binary values and are not scaled.

In [8]:
input_scaled = input_df.copy()

input_scaled[numeric_features] = scaler.transform(
    input_scaled[numeric_features]
)

input_scaled

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_L,Type_M
0,-0.95429,-0.950467,0.064337,0.279136,-1.692947,0,1


## 4. Failure Risk Prediction

The preprocessed sensor readings are passed to the trained Random Forest model.

The model returns the probability that the equipment belongs to the failure class.

In [10]:
failure_probability = model.predict_proba(input_scaled)[0, 1]

failure_prediction = model.predict(input_scaled)[0]

print("Failure Probability:", failure_probability)
print("Predicted Class:", failure_prediction)

Failure Probability: 0.0
Predicted Class: 0


## 5. Risk Classification

The predicted failure probability is converted into an operational risk level.

For the prototype, three risk levels are defined:

- LOW: failure probability below 20%
- MEDIUM: failure probability from 20% to below 50%
- HIGH: failure probability of 50% or greater

These thresholds are configurable and are intended as prototype alerting thresholds rather than validated industrial safety limits.

In [11]:
def get_risk_level(failure_probability):
    if failure_probability < 0.20:
        return "LOW"
    elif failure_probability < 0.50:
        return "MEDIUM"
    else:
        return "HIGH"

In [12]:
risk_level = get_risk_level(failure_probability)

print("Failure Probability:", failure_probability)
print("Risk Level:", risk_level)

Failure Probability: 0.0
Risk Level: LOW


## 6. Maintenance Recommendation

The monitoring system converts the predicted risk level into a maintenance recommendation.

The recommendations are intended to demonstrate the alerting workflow and should be refined using actual equipment-specific maintenance procedures in a production environment.

In [13]:
def get_maintenance_recommendation(risk_level):
    if risk_level == "LOW":
        return "Continue normal monitoring."
    elif risk_level == "MEDIUM":
        return "Schedule a maintenance inspection and monitor sensor trends closely."
    else:
        return "Immediate maintenance inspection recommended. Consider reducing equipment load if operational procedures permit."

In [14]:
recommendation = get_maintenance_recommendation(risk_level)

print("Risk Level:", risk_level)
print("Recommendation:", recommendation)

Risk Level: LOW
Recommendation: Continue normal monitoring.


## 7. Equipment Health Monitoring Function

The complete monitoring pipeline is encapsulated into a reusable function.

The function accepts raw machine sensor readings, applies the required preprocessing, generates a failure probability, assigns a risk level, and returns a maintenance recommendation.

In [15]:
def monitor_machine(machine_data):

    input_df = pd.DataFrame([machine_data])

    # Encode machine type
    input_df["Type_L"] = (input_df["Type"] == "L").astype(int)
    input_df["Type_M"] = (input_df["Type"] == "M").astype(int)

    input_df = input_df.drop(columns=["Type"])

    # Ensure the exact feature order
    input_df = input_df.reindex(
        columns=feature_columns,
        fill_value=0
    )

    # Scale numerical sensor features
    input_scaled = input_df.copy()

    input_scaled[numeric_features] = scaler.transform(
        input_scaled[numeric_features]
    )

    # Predict failure probability
    failure_probability = model.predict_proba(
        input_scaled
    )[0, 1]

    predicted_class = model.predict(
        input_scaled
    )[0]

    # Determine risk
    risk_level = get_risk_level(
        failure_probability
    )

    # Generate recommendation
    recommendation = get_maintenance_recommendation(
        risk_level
    )

    return {
        "failure_probability": failure_probability,
        "predicted_failure": int(predicted_class),
        "risk_level": risk_level,
        "recommendation": recommendation
    }

In [16]:
result = monitor_machine(sample_machine)

result

{'failure_probability': np.float64(0.0),
 'predicted_failure': 0,
 'risk_level': 'LOW',
 'recommendation': 'Continue normal monitoring.'}

## 8. Testing Different Equipment Conditions

The monitoring function is tested using several equipment conditions to verify that the system produces different failure probabilities and corresponding risk levels.

The test cases are illustrative scenarios for validating the prototype monitoring workflow.

In [17]:
low_risk_machine = {
    "Type": "M",
    "Air temperature [K]": 298.1,
    "Process temperature [K]": 308.6,
    "Rotational speed [rpm]": 1551,
    "Torque [Nm]": 42.8,
    "Tool wear [min]": 0
}

low_risk_result = monitor_machine(low_risk_machine)

low_risk_result

{'failure_probability': np.float64(0.0),
 'predicted_failure': 0,
 'risk_level': 'LOW',
 'recommendation': 'Continue normal monitoring.'}

In [18]:
medium_risk_machine = {
    "Type": "L",
    "Air temperature [K]": 300.5,
    "Process temperature [K]": 310.5,
    "Rotational speed [rpm]": 1450,
    "Torque [Nm]": 55.0,
    "Tool wear [min]": 150
}

medium_risk_result = monitor_machine(medium_risk_machine)

medium_risk_result

{'failure_probability': np.float64(0.013333333333333334),
 'predicted_failure': 0,
 'risk_level': 'LOW',
 'recommendation': 'Continue normal monitoring.'}

In [19]:
high_risk_machine = {
    "Type": "L",
    "Air temperature [K]": 301.5,
    "Process temperature [K]": 311.5,
    "Rotational speed [rpm]": 1300,
    "Torque [Nm]": 65.0,
    "Tool wear [min]": 200
}

high_risk_result = monitor_machine(high_risk_machine)

high_risk_result

{'failure_probability': np.float64(0.78),
 'predicted_failure': 1,
 'risk_level': 'HIGH',
 'recommendation': 'Immediate maintenance inspection recommended. Consider reducing equipment load if operational procedures permit.'}

In [20]:
test_results = pd.DataFrame([
    {
        "Machine": "Low Risk Test",
        **low_risk_result
    },
    {
        "Machine": "Elevated Risk Test",
        **medium_risk_result
    },
    {
        "Machine": "High Risk Test",
        **high_risk_result
    }
])

test_results

,Machine,failure_probability,predicted_failure,risk_level,recommendation
0,Low Risk Test,0.000000,0,LOW,Continue normal monitoring.
1,Elevated Risk Test,0.013333,0,LOW,Continue normal monitoring.
2,High Risk Test,0.780000,1,HIGH,Immediate maintenance inspection recommended. ...


## 9. Automatic Maintenance Alerts

The monitoring system generates an alert when the predicted equipment failure risk reaches the HIGH category.

LOW-risk equipment continues normal monitoring, while HIGH-risk equipment generates an actionable maintenance alert.

In [22]:
def generate_alert(monitoring_result):

    if monitoring_result["risk_level"] == "HIGH":
        return {
            "alert": True,
            "severity": "HIGH",
            "message": (
                "HIGH FAILURE RISK: "
                "Maintenance inspection required."
            )
        }

    elif monitoring_result["risk_level"] == "MEDIUM":
        return {
            "alert": True,
            "severity": "MEDIUM",
            "message": (
                "ELEVATED FAILURE RISK: "
                "Schedule inspection and continue close monitoring."
            )
        }

    else:
        return {
            "alert": False,
            "severity": "LOW",
            "message": "No maintenance alert required."
        }

In [23]:
high_risk_alert = generate_alert(high_risk_result)

high_risk_alert

{'alert': True,
 'severity': 'HIGH',
 'message': 'HIGH FAILURE RISK: Maintenance inspection required.'}

In [24]:
low_risk_alert = generate_alert(low_risk_result)

low_risk_alert

{'alert': False,
 'severity': 'LOW',
 'message': 'No maintenance alert required.'}

## 10. Complete Equipment Monitoring Pipeline

The monitoring pipeline combines failure prediction, risk classification, maintenance recommendations, and alert generation into a single reusable workflow.

In [25]:
def evaluate_machine(machine_data):

    monitoring_result = monitor_machine(machine_data)

    alert_result = generate_alert(monitoring_result)

    return {
        **monitoring_result,
        **alert_result
    }

In [26]:
high_risk_evaluation = evaluate_machine(high_risk_machine)

high_risk_evaluation

{'failure_probability': np.float64(0.78),
 'predicted_failure': 1,
 'risk_level': 'HIGH',
 'recommendation': 'Immediate maintenance inspection recommended. Consider reducing equipment load if operational procedures permit.',
 'alert': True,
 'severity': 'HIGH',
 'message': 'HIGH FAILURE RISK: Maintenance inspection required.'}

In [27]:
low_risk_evaluation = evaluate_machine(low_risk_machine)

low_risk_evaluation

{'failure_probability': np.float64(0.0),
 'predicted_failure': 0,
 'risk_level': 'LOW',
 'recommendation': 'Continue normal monitoring.',
 'alert': False,
 'severity': 'LOW',
 'message': 'No maintenance alert required.'}

In [28]:
final_test_results = pd.DataFrame([
    {
        "Machine": "Low Risk",
        **evaluate_machine(low_risk_machine)
    },
    {
        "Machine": "Elevated Risk",
        **evaluate_machine(medium_risk_machine)
    },
    {
        "Machine": "High Risk",
        **evaluate_machine(high_risk_machine)
    }
])

final_test_results

,Machine,failure_probability,predicted_failure,risk_level,recommendation,alert,severity,message
0,Low Risk,0.000000,0,LOW,Continue normal monitoring.,False,LOW,No maintenance alert required.
1,Elevated Risk,0.013333,0,LOW,Continue normal monitoring.,False,LOW,No maintenance alert required.
2,High Risk,0.780000,1,HIGH,Immediate maintenance inspection recommended. ...,True,HIGH,HIGH FAILURE RISK: Maintenance inspection requ...
